In [9]:
%pip install influxdb-client

  Using cached influxdb_client-1.36.1-py3-none-any.whl (721 kB)
  Using cached reactivex-4.0.4-py3-none-any.whl (217 kB)
Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install flightsql-dbapi

  Using cached flightsql_dbapi-0.2.1-py3-none-any.whl (23 kB)
  Using cached protobuf-4.23.1-cp37-abi3-manylinux2014_x86_64.whl (304 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.6 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.9
    Uninstalling SQLAlchemy-2.0.9:
      Successfully uninstalled SQLAlchemy-2.0.9
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ray 2.0.0 requires protobuf<4.0.0,>=3.15.3, but you have protobuf 4.23.1 which is incompatible.
mlflow 2.2.2 requires pytz<2023, but you have pytz 2023.3 which is incompatible.
mlflow-skinny 2.2.2 requires pytz<2023, but you have pytz 2023.3 which is incompatible.
azureml-mlflo

In [8]:
!pip install flightsql-dbapi

In [1]:
from flightsql import FlightSQLClient


#query data for the last 24 hours
query = """SELECT *
FROM 'production_data'
WHERE time >= now() - interval '30 days'
ORDER BY time ASC
"""

# Define the query client
query_client = FlightSQLClient(
  host = "us-east-1-1.aws.cloud2.influxdata.com",
  token = 'n2TK4tu9UOPRHB_wKnNSXhuVlmZRCUyhnD7ZHpLDAOv1zfTHCDi8JAx474vM-zQNhcHnoLYRbbUYpEQ19Ps9LQ==',
  metadata={"bucket-name": "rpi3test"})

# Execute the query
info = query_client.execute(query)
reader = query_client.do_get(info.endpoints[0].ticket)

# Convert to dataframe
data = reader.read_all()
df = data.to_pandas().sort_values(by="time")
print(df.head())


   ambient_temperature  dissolved_oxygen  do_volt  ec_volt   
0            38.632812          6.715018     1603     1593  \
1            38.632812          4.844862     1191     1726   
2            38.632812          5.882175     1446     1640   
3            38.632812          7.769678     1910     1592   
4            38.632812          4.551974     1119     1678   

   electrical_conductivity   humidity  ph_level  ph_voltge       sensor   
0                 8.393532  52.990723  4.171512       2002  raspberrypi  \
1                 8.951214  53.088379  6.414018       1604  raspberrypi   
2                 8.505209  53.088379  5.642101       1741  raspberrypi   
3                 8.256276  53.088379  6.707009       1552  raspberrypi   
4                 8.702281  53.088379  6.475997       1593  raspberrypi   

   temperature                          time  
0         33.5 2023-04-20 05:06:53.526554793  
1         34.5 2023-04-20 05:07:04.505995759  
2         34.5 2023-04-20 05:07:15.

In [5]:
df.to_csv('influxproductiondata.csv', index=False)

In [10]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# authenticate
credential = DefaultAzureCredential()

# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id="abb3353d-14ab-4405-8fec-2be226eecc91",
    resource_group_name="BARRIBAL.JOSHUAGEORGE-rg",
    workspace_name="aquagrow",
)

In [9]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
import time

# update the 'my_path' variable to match the location of where you downloaded the data on your
# local filesystem

my_path = "./influxproductiondata.csv"
# set the version number of the data asset to the current UTC time
v1 = time.strftime("%Y.%m.%d.%H%M%S", time.gmtime())


my_data = Data(
    name="production-dataset",
    version=v1,
    description="Aquagrow production data set",
    path=my_path,
    type=AssetTypes.URI_FILE,
)

# create data asset
ml_client.data.create_or_update(my_data)

print(f"Data asset created. Name: {my_data.name}, version: {my_data.version}")

ClientAuthenticationError: (UserError) Identity(object id: 8a2824e4-874d-4d9d-9485-805c45b3e24f) does not have permissions for Microsoft.MachineLearningServices/workspaces/datasets/registered/write actions. Please refer to https://aka.ms/azureml-auth-troubleshooting to fix the permissions issue.
Code: UserError
Message: Identity(object id: 8a2824e4-874d-4d9d-9485-805c45b3e24f) does not have permissions for Microsoft.MachineLearningServices/workspaces/datasets/registered/write actions. Please refer to https://aka.ms/azureml-auth-troubleshooting to fix the permissions issue.